# GENERAR PARQUET CON NIVEL OPTICO POR OLT

In [1]:
import pandas as pd  #Importar librería para manipulación de datos
import time  #Importar módulo para manejo de tiempo
import re  #Importar librería para expresiones regulares
import os  #Importar módulo para manejo de sistema de archivos
import json  #Importar módulo para lectura de JSON
import calendar  #Importar módulo para manejo de calendarios
from datetime import datetime  #Importar formateo de fechas
from sqlalchemy import create_engine, text  #Importar conector SQL

# Cargar configuración desde JSON
with open('config_etl_zabbix.json', 'r') as file:  #Abrir archivo JSON
    config = json.load(file)  #Cargar datos del JSON

DB_USER = config['database']['user']  #Definir usuario desde JSON
DB_PASS = config['database']['password']  #Definir contraseña desde JSON
DB_HOST = config['database']['host']  #Definir host desde JSON
DB_NAME = config['database']['name']  #Definir nombre DB desde JSON

extr_anio = config['etl_params']['anio']  #Definir año a procesar desde JSON
extr_mes = config['etl_params']['mes']  #Definir mes a procesar desde JSON
LISTA_OLTS = config['etl_params']['lista_olts']  #Definir lista de OLTs desde JSON

RUTA_SALIDA = "datalake/bronze/zabbix_final"  #Definir ruta de salida
if not os.path.exists(RUTA_SALIDA):  #Verificar existencia de directorio
    os.makedirs(RUTA_SALIDA)  #Crear directorio si no existe

#Conexión
str_conn = f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}'  #Construir string de conexión
engine = create_engine(str_conn)  #Inicializar motor de base de datos

def extraer_ont_id(txt):  #Definir función de extracción de ID
    match = re.search(r'ONT\s+(\d+|[A-Z0-9_]+)', txt, re.IGNORECASE)  #Buscar patrón de ID en texto
    return match.group(1) if match else None  #Retornar ID encontrado o nulo

def pipeline_olt(nombre_olt):  #Definir función de procesamiento por OLT
    print(f"Procesando: {nombre_olt} para {extr_anio}-{extr_mes:02d}")  #Imprimir inicio de proceso
    
    try:  #Iniciar bloque de manejo de errores
        with engine.connect() as conn:  #Establecer conexión a base de datos
            #A.Obtener HostID
            res = conn.execute(text(f"SELECT hostid FROM hosts WHERE name = '{nombre_olt}'")).fetchone()  #Consultar ID de host
            if not res:  #Verificar si existe host
                print("   OLT no encontrada.")  #Imprimir mensaje de error
                return  #Finalizar ejecución para esta OLT
            host_id = res[0]  #Asignar ID de host

            print("   Obteniendo Potencia Óptica (Trends)...")  #Imprimir paso de extracción
            
            #Calcular timestamps dinámicamente para el mes seleccionado
            _, ultimo_dia = calendar.monthrange(extr_anio, extr_mes)  #Obtener último día del mes
            ts_start_trends = int(time.mktime(datetime(extr_anio, extr_mes, 1).timetuple()))  #Definir timestamp inicio mes
            ts_end_trends = int(time.mktime(datetime(extr_anio, extr_mes, ultimo_dia, 23, 59, 59).timetuple()))  #Definir timestamp fin mes
            
            q_opt = f"""
            SELECT i.name, AVG(t.value_avg) as rx_avg 
            FROM items i JOIN trends t ON i.itemid = t.itemid
            WHERE i.hostid = {host_id} AND t.clock BETWEEN {ts_start_trends} AND {ts_end_trends}
            AND i.key_ LIKE 'OLT.client.ONT.Rx.power%%'
            GROUP BY i.name
            """  #Definir consulta SQL
            
            resultado = conn.execute(text(q_opt))  #Ejecutar consulta
            filas = resultado.fetchall()  #Extraer todas las filas
            
            if not filas:  #Si la lista de filas está vacía
                print("   Sin datos ópticos para este periodo.")  #Imprimir mensaje de advertencia
                return  #Finalizar ejecución para esta OLT

            df_opt = pd.DataFrame(filas, columns=resultado.keys())  #Crear dataframe con nombres de columnas

            df_opt['ont_id'] = df_opt['name'].apply(extraer_ont_id)  #Aplicar extracción de ID
            df_final = df_opt.groupby('ont_id')['rx_avg'].mean().reset_index()  #Agrupar por ID y promediar

            df_opt['ont_id'] = df_opt['name'].apply(extraer_ont_id)  #Aplicar extracción de ID
            df_final = df_opt.groupby('ont_id')['rx_avg'].mean().reset_index()  #Agrupar por ID y promediar

            #Construir el formato YYMM (ej. 2024, mes 2 -> 2402)
            str_yymm = datetime(extr_anio, extr_mes, 1).strftime('%y%m')  #Formatear fecha
            archivo = f"{RUTA_SALIDA}/{nombre_olt}_optical_only_{str_yymm}.parquet"  #Definir nombre de archivo final
            
            df_final.to_parquet(archivo, index=False)  #Guardar dataframe en formato parquet
            print(f"   Guardado {archivo}")  #Imprimir confirmación de guardado

    except Exception as e:  #Capturar excepciones
        print(f"   Error {e}")  #Imprimir mensaje de error

#Ejecutar
for olt in LISTA_OLTS:  #Iterar sobre lista de OLTs
    pipeline_olt(olt)  #Ejecutar pipeline para cada OLT
print("\nPipeline Terminado.")  #Imprimir fin de proceso general

Procesando: VIRREY_OLT01 para 2026-02
   Error (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '192.168.253.241' ([WinError 10013] Intento de acceso a un socket no permitido por sus permisos de acceso)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Procesando: BELO_OLT01 para 2026-02
   Error (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '192.168.253.241' ([WinError 10013] Intento de acceso a un socket no permitido por sus permisos de acceso)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Procesando: MADERO_OLT01 para 2026-02
   Error (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '192.168.253.241' ([WinError 10013] Intento de acceso a un socket no permitido por sus permisos de acceso)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Procesando: PERIODISMO_OLT01 para 2026-02
   Error (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '192.168.253

# MOSTRAR PARQUET CON NIVEL OPTICO POR OLT

In [2]:
import pandas as pd
import os
import glob

#Definir ruta de almacenamiento de archivos parquet
RUTA_DATALAKE = "datalake/bronze/zabbix_final"

#Buscar todos los archivos .parquet en la carpeta especificada
archivos_parquet = glob.glob(f"{RUTA_DATALAKE}/*.parquet")

print(f"Se encontraron {len(archivos_parquet)} archivos en el Data Lake.\n")

if not archivos_parquet:
    print("ALERTA: No hay archivos. Revisa si el script anterior terminó correctamente.")
else:
    for archivo in archivos_parquet:
        nombre_archivo = os.path.basename(archivo)
        print("=" * 80)
        print(f"AUDITANDO: {nombre_archivo}")
        
        try:
            #Cargar datos del archivo parquet
            df = pd.read_parquet(archivo)
            
            #Calcular métricas de volumen
            cant_registros = len(df)
            cant_columnas = len(df.columns)
            print(f"   Dimensiones: {cant_registros} ONTs (filas) x {cant_columnas} Variables (columnas)")
            
            #Validar calidad de datos
            #Verificar existencia de nulos en columnas clave
            nulos_rx = df['rx_avg'].isnull().sum()
            
            print(f"   Calidad:")
            print(f"      - Nulos en Potencia (Rx): {nulos_rx}")
            
            #Validar lógica de negocio
            print(f"   Lógica de Negocio:")
            print(f"      - Potencia Promedio: {df['rx_avg'].mean():.2f} dBm (Debería ser entre -15 y -27)")

            #Mostrar primeros 5 registros de muestra
            print("\n   VISTA PREVIA (Top 5):")
            #Seleccionar columnas clave para visualización ordenada
            cols_mostrar = [c for c in ['ont_id', 'rx_avg'] if c in df.columns]
            print(df[cols_mostrar].head(5).to_string(index=False))
            print("\n")

        except Exception as e:
            print(f"   Error leyendo archivo: {e}")

print("=" * 80)
print("Auditoría Finalizada.")

Se encontraron 24 archivos en el Data Lake.

AUDITANDO: ATAPANEO_OLT01_optical_only_2601.parquet
   Dimensiones: 27 ONTs (filas) x 2 Variables (columnas)
   Calidad:
      - Nulos en Potencia (Rx): 0
   Lógica de Negocio:
      - Potencia Promedio: -25.96 dBm (Debería ser entre -15 y -27)

   VISTA PREVIA (Top 5):
ont_id     rx_avg
 45390 -18.010035
 45391 -40.000000
 45400 -20.477984
 45406 -40.000000
 45408 -24.232640


AUDITANDO: ATAPANEO_OLT01_optical_only_2602.parquet
   Dimensiones: 30 ONTs (filas) x 2 Variables (columnas)
   Calidad:
      - Nulos en Potencia (Rx): 0
   Lógica de Negocio:
      - Potencia Promedio: -26.02 dBm (Debería ser entre -15 y -27)

   VISTA PREVIA (Top 5):
ont_id     rx_avg
 45390 -17.784994
 45391 -40.000000
 45400 -20.225799
 45406 -40.000000
 45408 -26.907175


AUDITANDO: BELO_OLT01_optical_only_2601.parquet
   Dimensiones: 823 ONTs (filas) x 2 Variables (columnas)
   Calidad:
      - Nulos en Potencia (Rx): 0
   Lógica de Negocio:
      - Potencia Pr

   Dimensiones: 741 ONTs (filas) x 2 Variables (columnas)
   Calidad:
      - Nulos en Potencia (Rx): 0
   Lógica de Negocio:
      - Potencia Promedio: -28.25 dBm (Debería ser entre -15 y -27)

   VISTA PREVIA (Top 5):
ont_id     rx_avg
  1000 -25.632136
   103 -40.000000
  1044 -32.698680
  1045 -40.000000
  1085 -27.071387


AUDITANDO: SANTIAGUITO_OLT01_optical_only_2601.parquet
   Dimensiones: 648 ONTs (filas) x 2 Variables (columnas)
   Calidad:
      - Nulos en Potencia (Rx): 0
   Lógica de Negocio:
      - Potencia Promedio: -28.09 dBm (Debería ser entre -15 y -27)

   VISTA PREVIA (Top 5):
ont_id     rx_avg
   107 -20.289367
  1076 -24.532436
  1161 -24.383485
  1164 -20.356896
  1168 -20.632507


AUDITANDO: SANTIAGUITO_OLT01_optical_only_2602.parquet
   Dimensiones: 664 ONTs (filas) x 2 Variables (columnas)
   Calidad:
      - Nulos en Potencia (Rx): 0
   Lógica de Negocio:
      - Potencia Promedio: -28.12 dBm (Debería ser entre -15 y -27)

   VISTA PREVIA (Top 5):
ont_id    